In [1]:
import sys

# If running in a fresh environment, uncomment this:
# !"{sys.executable}" -m pip install -q catboost scikit-learn pandas numpy

In [2]:
import pandas as pd

In [3]:
from pathlib import Path

DATA_PATH_CANDIDATES = [
    Path("property_listings.csv"),
    Path("data") / "property_listings.csv",
]
DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find 'property_listings.csv'. Place it next to this notebook, "
        "or at 'data/property_listings.csv'."
    )

df = pd.read_csv(DATA_PATH)

In [4]:
df.head()

,zpid,price,homeStatus,homeType,datePosted,streetAddress,city,state,zipcode,county,...,rentZestimate,bathrooms,bedrooms,pageViewCount,favoriteCount,propertyTaxRate,timeOnZillow,dateSold,url,lastUpdated
0,32107262.0,750000.0,Recently Sold,Multi Family,2024-03-19,7417 87th Rd,Jamaica,NY,11421.0,Queens County,...,2930.0,2.0,NaN,20.0,0.0,0.86,9 hours,2024-11-24,https://www.zillow.com/homedetails/7417-87th-R...,2024-11-25 09:04:11.007468 UTC
1,20503342.0,3995.0,Recently Sold,Apartment,2024-09-24,1300 Midvale Ave APT 510,Los Angeles,CA,90024.0,Los Angeles County,...,3867.0,2.0,2.0,187.0,5.0,1.16,9 hours,2024-11-24,https://www.zillow.com/homedetails/1300-Midval...,2024-11-25 09:04:11.007468 UTC
2,20183958.0,820000.0,Recently Sold,Single Family,2024-10-27,8300 Capps Ave,Northridge,CA,91324.0,Los Angeles County,...,4540.0,2.0,3.0,21.0,0.0,1.16,9 hours,2024-11-24,https://www.zillow.com/homedetails/8300-Capps-...,2024-11-25 09:04:11.007468 UTC
3,32332472.0,550000.0,Recently Sold,Single Family,2024-07-09,433 Hamden Ave,Staten Island,NY,10306.0,Richmond County,...,2668.0,1.0,2.0,96.0,0.0,0.89,9 hours,2024-11-24,https://www.zillow.com/homedetails/433-Hamden-...,2024-11-25 09:04:11.007468 UTC
4,352427429.0,703478.0,Recently Sold,Single Family,2024-06-19,504 Edwin St #8,Nashville,TN,37207.0,Davidson County,...,3599.0,4.0,4.0,7.0,0.0,0.57,9 hours,2024-11-24,https://www.zillow.com/homedetails/504-Edwin-S...,2024-11-25 09:04:11.007468 UTC


In [5]:
df.columns

Index(['zpid', 'price', 'homeStatus', 'homeType', 'datePosted',
       'streetAddress', 'city', 'state', 'zipcode', 'county', 'yearBuilt',
       'livingArea', 'livingAreaUnits', 'rentZestimate', 'bathrooms',
       'bedrooms', 'pageViewCount', 'favoriteCount', 'propertyTaxRate',
       'timeOnZillow', 'dateSold', 'url', 'lastUpdated'],
      dtype='object')

In [6]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def _parse_time_on_zillow_hours(x: object) -> float:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    m = re.search(r"(\d+(?:\.\d+)?)\s*(minute|minutes|min|hour|hours|hr|hrs|day|days|week|weeks)", s)
    if not m:
        return np.nan
    value = float(m.group(1))
    unit = m.group(2)
    if unit.startswith("min") or unit.startswith("minute"):
        return value / 60.0
    if unit.startswith("hr") or unit.startswith("hour"):
        return value
    if unit.startswith("day"):
        return value * 24.0
    if unit.startswith("week"):
        return value * 24.0 * 7.0
    return np.nan


def add_date_features(df_in: pd.DataFrame, date_cols: list[str]) -> pd.DataFrame:
    df = df_in.copy()
    for c in date_cols:
        if c not in df.columns:
            continue
        dt = pd.to_datetime(df[c], errors="coerce")
        df[f"{c}_year"] = dt.dt.year
        df[f"{c}_month"] = dt.dt.month
        df[f"{c}_day"] = dt.dt.day
        df[f"{c}_dayofweek"] = dt.dt.dayofweek
        df[f"{c}_is_month_start"] = dt.dt.is_month_start.astype("float")
        df[f"{c}_is_month_end"] = dt.dt.is_month_end.astype("float")
        df.drop(columns=[c], inplace=True)
    return df


TARGET = "price"
DROP_COLS = ["url", "zpid", "ig"]  # drop if present
DATE_COLS = ["datePosted", "dateSold", "lastUpdated"]

# Expect `df` and `DATA_PATH` from earlier cells; fall back if running this cell alone.
if "df" not in globals():
    from pathlib import Path

    DATA_PATH_CANDIDATES = [
        Path("property_listings.csv"),
        Path("data") / "property_listings.csv",
    ]
    DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
    if DATA_PATH is None:
        raise FileNotFoundError(
            "Could not find 'property_listings.csv'. Place it next to this notebook, "
            "or at 'data/property_listings.csv'."
        )

    df = pd.read_csv(DATA_PATH)

# Work on a copy
work_df = df.copy()

# Basic cleanup
if TARGET not in work_df.columns:
    raise ValueError(f"Expected target column '{TARGET}' not found. Columns: {list(work_df.columns)}")

work_df = work_df.drop(columns=[c for c in DROP_COLS if c in work_df.columns])

# Feature engineering
if "timeOnZillow" in work_df.columns:
    work_df["timeOnZillow_hours"] = work_df["timeOnZillow"].map(_parse_time_on_zillow_hours)
    work_df = work_df.drop(columns=["timeOnZillow"])

work_df = add_date_features(work_df, DATE_COLS)

# Split
X = work_df.drop(columns=[TARGET])
X = X.replace([np.inf, -np.inf], np.nan)

y = pd.to_numeric(work_df[TARGET], errors="coerce").replace([np.inf, -np.inf], np.nan)

# CatBoost doesn't allow NaN targets for RMSE/MAE metrics
keep = y.notna()
X = X.loc[keep].copy()
y = y.loc[keep].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# CatBoost can natively handle categorical columns, but they must be string/int (no NaN floats)
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]

for c in cat_cols:
    X_train[c] = X_train[c].astype("string").fillna("__MISSING__")
    X_test[c] = X_test[c].astype("string").fillna("__MISSING__")

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    early_stopping_rounds=100,
    verbose=200,
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True,
)

pred = model.predict(X_test)

mse = mean_squared_error(y_test, pred)
rmse = float(np.sqrt(mse))
mae = float(mean_absolute_error(y_test, pred))
r2 = float(r2_score(y_test, pred))

print({"rmse": float(rmse), "mae": float(mae), "r2": float(r2), "n_cat_cols": len(cat_cols)})

# Save artifacts
out_dir = Path("models")
out_dir.mkdir(parents=True, exist_ok=True)
model_path = out_dir / "catboost_price.cbm"
model.save_model(model_path)

pd.Series(cat_cols).to_csv(out_dir / "cat_features.csv", index=False, header=["cat_feature"])

0:	learn: 4019996.1244187	test: 1027647.9758039	best: 1027647.9758039 (0)	total: 180ms	remaining: 9m 1s
200:	learn: 1203957.9941693	test: 810236.4952706	best: 808663.1801916 (193)	total: 8.69s	remaining: 2m 1s
400:	learn: 704737.5520995	test: 788383.6338822	best: 788383.6338822 (400)	total: 16.9s	remaining: 1m 49s
600:	learn: 548316.2098525	test: 781220.3948256	best: 781220.3948256 (600)	total: 25.4s	remaining: 1m 41s
800:	learn: 403871.4264354	test: 773328.2763970	best: 773212.7222384 (787)	total: 34.4s	remaining: 1m 34s
1000:	learn: 343236.3485340	test: 771660.0686099	best: 771008.9182953 (952)	total: 43.7s	remaining: 1m 27s
1200:	learn: 303226.5022962	test: 770336.5744815	best: 770103.9074459 (1191)	total: 52.6s	remaining: 1m 18s
1400:	learn: 269273.3836676	test: 770096.1905146	best: 769808.0160528 (1353)	total: 1m 1s	remaining: 1m 9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 769808.0161
bestIteration = 1353

Shrink model to first 1354 iterations.
{'rmse': 7

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor


def _make_ohe() -> OneHotEncoder:
    # sklearn changed `sparse` -> `sparse_output`
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def _build_preprocess_for_trees(X: pd.DataFrame) -> ColumnTransformer:
    cat_cols_local = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith("string")]
    num_cols_local = [c for c in X.columns if c not in cat_cols_local]

    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                num_cols_local,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("ohe", _make_ohe()),
                    ]
                ),
                cat_cols_local,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def _build_preprocess_for_nn(X: pd.DataFrame) -> ColumnTransformer:
    cat_cols_local = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith("string")]
    num_cols_local = [c for c in X.columns if c not in cat_cols_local]

    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                num_cols_local,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("ohe", _make_ohe()),
                    ]
                ),
                cat_cols_local,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def eval_model(name: str, model_pipeline, X_train, y_train, X_test, y_test) -> dict:
    model_pipeline.fit(X_train, y_train)
    p = model_pipeline.predict(X_test)

    mse_local = mean_squared_error(y_test, p)
    rmse_local = float(np.sqrt(mse_local))
    mae_local = float(mean_absolute_error(y_test, p))
    r2_local = float(r2_score(y_test, p))

    return {"model": name, "rmse": rmse_local, "mae": mae_local, "r2": r2_local}

In [8]:
# Collect results (CatBoost + sklearn models)
results = []

# From the CatBoost cell above (rmse/mae/r2 variables)
if all(k in globals() for k in ["rmse", "mae", "r2"]):
    results.append({"model": "CatBoost", "rmse": float(rmse), "mae": float(mae), "r2": float(r2)})

preprocess_tree = _build_preprocess_for_trees(X_train)
preprocess_nn = _build_preprocess_for_nn(X_train)

In [9]:
# 1) Boosted trees (sklearn)
boosted = Pipeline(
    steps=[
        ("preprocess", preprocess_tree),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="squared_error",
                learning_rate=0.05,
                max_depth=None,
                max_iter=600,
                random_state=42,
            ),
        ),
    ]
)

results.append(eval_model("BoostedTrees(HistGB)", boosted, X_train, y_train, X_test, y_test))
results[-1]

{'model': 'BoostedTrees(HistGB)',
 'rmse': 917178.676758826,
 'mae': 275374.45825369906,
 'r2': 0.22383836047970096}

In [10]:
# 2) Random forest
rf = Pipeline(
    steps=[
        ("preprocess", preprocess_tree),
        (
            "model",
            RandomForestRegressor(
                n_estimators=600,
                random_state=42,
                n_jobs=-1,
                max_features="sqrt",
            ),
        ),
    ]
)

results.append(eval_model("RandomForest", rf, X_train, y_train, X_test, y_test))
results[-1]

{'model': 'RandomForest',
 'rmse': 780628.642697607,
 'mae': 214608.3069315761,
 'r2': 0.43774511095105006}

In [11]:
# 3) Neural net (MLP)
# Note: MLP is sensitive to feature scaling, hence preprocess_nn
nn = Pipeline(
    steps=[
        ("preprocess", preprocess_nn),
        (
            "model",
            MLPRegressor(
                hidden_layer_sizes=(256, 128, 64),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=400,
                random_state=42,
                early_stopping=True,
                n_iter_no_change=20,
                verbose=False,
            ),
        ),
    ]
)

results.append(eval_model("NeuralNet(MLP)", nn, X_train, y_train, X_test, y_test))
results[-1]

{'model': 'NeuralNet(MLP)',
 'rmse': 902542.8704666372,
 'mae': 287491.2243053953,
 'r2': 0.24841179535140068}

In [12]:
# 4) Ensemble (average of boosted trees + RF + NN)
# We use pipelines directly as estimators.
ensemble = VotingRegressor(
    estimators=[
        ("boosted", boosted),
        ("rf", rf),
        ("nn", nn),
    ]
)

results.append(eval_model("Ensemble(VotingReg)", ensemble, X_train, y_train, X_test, y_test))
results[-1]

{'model': 'Ensemble(VotingReg)',
 'rmse': 814116.4409463997,
 'mae': 233843.4702809245,
 'r2': 0.3884706204448859}

In [13]:
# Comparison table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="rmse", ascending=True).reset_index(drop=True)
results_df

,model,rmse,mae,r2
0,CatBoost,769808.016053,216991.022036,0.453224
1,RandomForest,780628.642698,214608.306932,0.437745
2,Ensemble(VotingReg),814116.440946,233843.470281,0.388471
3,NeuralNet(MLP),902542.870467,287491.224305,0.248412
4,BoostedTrees(HistGB),917178.676759,275374.458254,0.223838


In [ ]:
from catboost import Pool, cv

# Prepare CatBoost Pools for tuning
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

# Ensure categorical cols are safe for CatBoost (string + no NaN)
cat_cols_cb = [c for c in X_train_cb.columns if X_train_cb[c].dtype == "object" or str(X_train_cb[c].dtype).startswith("string")]
for c in cat_cols_cb:
    X_train_cb[c] = X_train_cb[c].astype("string").fillna("__MISSING__")
    X_test_cb[c] = X_test_cb[c].astype("string").fillna("__MISSING__")

train_pool = Pool(X_train_cb, y_train, cat_features=cat_cols_cb)


In [ ]:
# Random search with CatBoost CV
rng = np.random.default_rng(42)

def sample_params() -> dict:
    return {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": 4000,
        "learning_rate": float(rng.choice([0.02, 0.03, 0.05, 0.08, 0.1])),
        "depth": int(rng.integers(6, 11)),
        "l2_leaf_reg": float(rng.choice([1.0, 3.0, 5.0, 8.0, 12.0, 20.0, 30.0])),
        "random_strength": float(rng.choice([0.0, 0.5, 1.0, 2.0, 5.0, 10.0])),
        "bagging_temperature": float(rng.choice([0.0, 0.2, 0.5, 1.0, 2.0, 5.0])),
        "rsm": float(rng.choice([0.7, 0.8, 0.9, 1.0])),
        "min_data_in_leaf": int(rng.choice([1, 5, 10, 20, 50, 100])),
        "bootstrap_type": str(rng.choice(["Bayesian", "Bernoulli", "MVS"])),
        "random_seed": 42,
        "verbose": False,
    }


def cv_score(params: dict) -> tuple[float, int]:
    cv_res = cv(
        pool=train_pool,
        params={
            **params,
            "od_type": "Iter",
            "od_wait": 100,
        },
        fold_count=3,
        shuffle=True,
        partition_random_seed=42,
        early_stopping_rounds=100,
        verbose=False,
    )
    # Columns typically: 'test-RMSE-mean', 'test-RMSE-std', ...
    best_idx = int(cv_res["test-RMSE-mean"].idxmin())
    best_rmse = float(cv_res.loc[best_idx, "test-RMSE-mean"])
    best_iter = best_idx + 1
    return best_rmse, best_iter


n_trials = 25
search_rows = []

for i in range(n_trials):
    p = sample_params()
    rmse_cv, best_iter = cv_score(p)
    search_rows.append({"trial": i, "cv_rmse": rmse_cv, "best_iter": best_iter, **p})
    print(f"trial={i:02d} cv_rmse={rmse_cv:.4f} best_iter={best_iter} params={{lr={p['learning_rate']}, depth={p['depth']}, l2={p['l2_leaf_reg']}}}")

cb_search_df = pd.DataFrame(search_rows).sort_values("cv_rmse", ascending=True).reset_index(drop=True)
cb_search_df.head(10)

In [ ]:
# Train the best-tuned CatBoost model on train, evaluate on test
best_row = cb_search_df.iloc[0].to_dict()
best_params = {k: best_row[k] for k in best_row.keys() if k in {
    "loss_function",
    "eval_metric",
    "learning_rate",
    "depth",
    "l2_leaf_reg",
    "random_strength",
    "bagging_temperature",
    "rsm",
    "min_data_in_leaf",
    "bootstrap_type",
    "random_seed",
}}

best_iterations = int(best_row["best_iter"])

cb_tuned = CatBoostRegressor(
    **best_params,
    iterations=best_iterations,
    verbose=200,
)

cb_tuned.fit(
    X_train_cb,
    y_train,
    cat_features=cat_cols_cb,
    eval_set=(X_test_cb, y_test),
    use_best_model=False,
)

pred_tuned = cb_tuned.predict(X_test_cb)
rmse_tuned = float(np.sqrt(mean_squared_error(y_test, pred_tuned)))
mae_tuned = float(mean_absolute_error(y_test, pred_tuned))
r2_tuned = float(r2_score(y_test, pred_tuned))

print({"rmse": rmse_tuned, "mae": mae_tuned, "r2": r2_tuned, "iterations": best_iterations})

# Save tuned model
out_dir = Path("models")
out_dir.mkdir(parents=True, exist_ok=True)
cb_tuned.save_model(out_dir / "catboost_price_tuned.cbm")

# Add to comparison table
results.append({"model": "CatBoost(tuned)", "rmse": rmse_tuned, "mae": mae_tuned, "r2": r2_tuned})
results_df = pd.DataFrame(results).sort_values(by="rmse", ascending=True).reset_index(drop=True)
results_df